# Chapter 5: Production Workflows — Pipelines, Cross-Validation & Model Persistence

### Why This Matters

Everything so far has been demonstrated as separate, manually-sequenced steps. Real, deployable ML work bundles preprocessing and modeling into a single object — a **Pipeline** — both for cleanliness and, more importantly, to structurally prevent the data leakage warned about in Chapter 4.

### 5.1 Building a Pipeline

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])
pipe.fit(X_train, y_train)          # fits the scaler on X_train, THEN fits the model on the scaled result
proba = pipe.predict_proba(X_test)[:, 1]
print('ROC AUC:', roc_auc_score(y_test, proba))

A `Pipeline` is a named sequence of steps, each with a `.fit()`/`.transform()` (for preprocessing steps) or `.fit()`/`.predict()` (for the final model step). Calling `pipe.fit(X_train, y_train)` runs every step in order, automatically fitting each preprocessing step *only* on the data flowing through it at that point in the pipeline (correctly restricted to training data) — this is precisely why Pipelines are the professionally correct way to combine preprocessing and modeling, rather than manually calling `.fit_transform()` and `.fit()` as separate steps where it's easy to accidentally leak test data into a preprocessing fit.

💡 A Pipeline also dramatically simplifies deployment: instead of remembering to separately apply the same scaler, encoder, and model in the same order every time you get new data, you save and load *one* object that does everything internally, correctly, every time.

### 5.2 Cross-Validation — A More Reliable Performance Estimate

In [ ]:
scores = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
print(scores)
print("Mean ROC-AUC:", scores.mean())

A single train/test split gives you one performance number that depends partly on the luck of exactly which rows landed in the test set. **K-fold cross-validation** splits the data into `k` (commonly 5 or 10) roughly equal chunks ("folds"), trains on `k-1` of them and tests on the remaining one, and repeats this `k` times so every fold serves as the test set exactly once — giving you `k` performance estimates whose mean and spread tell you far more about how reliably the model performs than any single split could. 💡 Passing the whole `pipe` (not just the bare model) to `cross_val_score` is important — it re-fits the *entire* pipeline, scaler included, fresh within each fold, correctly avoiding leakage across every single fold rather than just the one final train/test split.

### 5.3 Systematic Hyperparameter Tuning: Grid Search

In [ ]:
param_grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [5, 10, 20]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

**Hyperparameters** are the settings you choose *before* training (like `max_depth` for a tree, or `n_clusters` for K-Means) as opposed to **parameters**, which the model learns from data during training (like a regression's coefficients). `GridSearchCV` automates the tedious "try every combination and see what performs best" process: it exhaustively trains and cross-validates a model for every combination in `param_grid` (here, 4 × 3 = 12 combinations, each evaluated with 5-fold CV, for 60 total model fits) and reports the best-performing combination. This exact idea — but far more efficient at scale — is what Volume 06's Optuna chapter automates with a smarter search strategy than brute-force grid search.

### 5.4 Persisting a Trained Model

In [ ]:
import joblib

joblib.dump(pipe, "churn_model.joblib")            # save the entire fitted pipeline to disk

loaded_pipe = joblib.load("churn_model.joblib")     # load it back later — no retraining needed
predictions = loaded_pipe.predict(X_test)

`joblib` (preferred over Python's general-purpose `pickle` for scikit-learn objects, since it's more efficient for the large NumPy arrays typically inside a fitted model) is how a trained model moves from your notebook into a real application — you train once, save the file, and any other script or service (a Streamlit app in Volume 10, a scheduled batch job, an API endpoint) simply loads the saved file and calls `.predict()`, without ever needing to retrain or even have access to the original training data.

### Cheat Sheet — Production ML

| Task | Code |
|---|---|
| Build a pipeline | `Pipeline([('scale', StandardScaler()), ('model', LogisticRegression())])` |
| Cross-validate | `cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')` |
| Tune hyperparameters | `GridSearchCV(model, param_grid, cv=5, scoring=...)` |
| Save a model | `joblib.dump(pipe, "model.joblib")` |
| Load a model | `joblib.load("model.joblib")` |

### 🎯 Business Challenge — Volume 05 Capstone Preview

*Scenario (from the source material, expanded):* a retail bank wants to predict which existing customers will accept a personal loan offer.

In [ ]:
np.random.seed(10)
n = 500
bank = pd.DataFrame({
    "Age": np.random.randint(23, 65, n),
    "Income": np.random.uniform(25, 200, n),          # ₹ thousands/year
    "Family_Size": np.random.randint(1, 5, n),
    "CCAvg": np.random.uniform(0.5, 10, n),            # avg credit card spend, ₹ thousands/month
    "Mortgage": np.random.choice([0, 0, 0, 150, 250, 400], n)
})
score = (-6 + 0.03*bank["Income"] + 0.4*bank["CCAvg"] - 0.15*bank["Family_Size"]
         + 0.002*bank["Mortgage"])
bank["Loan_Accepted"] = (np.random.rand(n) < 1/(1+np.exp(-score))).astype(int)

X = bank[["Age", "Income", "Family_Size", "CCAvg", "Mortgage"]]
y = bank["Loan_Accepted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipe_lr = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
pipe_lr.fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)

for name, m, needs_scale in [("Logistic Regression", pipe_lr, True), ("Decision Tree", tree, False)]:
    preds = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))

importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop drivers of loan acceptance (Decision Tree):\n", importance)

*Executive summary template (five bullets, filled in from your actual run's numbers):*
- We built two models — a Logistic Regression (interpretable, ROC-AUC ≈ X) and a Decision Tree (ROC-AUC ≈ Y) — to predict which customers will accept a personal loan offer.
- The top predictors of loan acceptance are [top 2–3 features from `feature_importances_`].
- [Income/CCAvg] has the strongest positive relationship with acceptance — customers with [higher X] are meaningfully more likely to accept.
- We recommend targeting the top-decile customers by predicted probability for the next loan campaign, which the model estimates will substantially outperform an untargeted mailing.
- Next step: validate this on a genuinely held-out future month of data before committing marketing budget, and consider a randomized pilot to establish a true causal lift (per Volume 04's correlation-vs-causation caution).

---

## Volume 05 — What You Should Be Able to Do Now

- Explain, in plain language, the difference between regression, classification, and clustering, and give a business example of each
- Correctly split data into train/test sets and explain why this prevents overfitting
- Fit a logistic regression and decision tree, and correctly interpret a confusion matrix and ROC-AUC for a business audience
- Explain why scaling must be fit only on training data, and build a scikit-learn `Pipeline` that does this automatically
- Use cross-validation and `GridSearchCV` to get a reliable performance estimate and tune a model responsibly
- Save and reload a trained model with `joblib`

**Next: `06_Gradient_Boosting_and_Optuna.md`**